# Demo 02: Industrial Shortcut Trap

## Learning goals
- show a classifier that looks accurate because it keys off a corner stamp rather than the central part;
- make the failure visible with direct evidence overlays and stamp swaps;
- compare the shortcut model with a part-focused intervention.

## Why this demo matters
Industrial pipelines often contain fixture marks, stickers, borders, and camera artefacts that correlate with the label for accidental reasons. This notebook keeps the whole failure mode inspectable in one place.
## Data source
This notebook is self-contained. Every image used below is generated inside the notebook; no external dataset files are read.



### Notebook display helpers

In [ ]:
import math

import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont

try:
    from IPython.display import display
except Exception:
    def display(obj):
        return None

FONT = ImageFont.load_default()


def np_to_pil(array):
    if isinstance(array, Image.Image):
        return array
    if array.dtype != np.uint8:
        array = (np.clip(array, 0.0, 1.0) * 255).astype(np.uint8)
    return Image.fromarray(array)


def titled(image, title, *, width=None):
    base = np_to_pil(image).convert('RGB')
    if width is not None and base.width != width:
        base = base.resize((width, round(base.height * width / base.width)))
    title_height = 22
    panel = Image.new('RGB', (base.width, base.height + title_height), (255, 255, 255))
    panel.paste(base, (0, title_height))
    draw = ImageDraw.Draw(panel)
    draw.rectangle((0, 0, base.width, title_height), fill=(245, 245, 245))
    draw.text((6, 5), title, fill=(20, 20, 20), font=FONT)
    return panel


def montage(panels, *, cols=3, padding=8, bg=(255, 255, 255)):
    prepared = [np_to_pil(panel).convert('RGB') for panel in panels]
    if not prepared:
        return Image.new('RGB', (320, 120), bg)
    cell_w = max(panel.width for panel in prepared)
    cell_h = max(panel.height for panel in prepared)
    rows = math.ceil(len(prepared) / cols)
    canvas = Image.new(
        'RGB',
        (cols * cell_w + (cols + 1) * padding, rows * cell_h + (rows + 1) * padding),
        bg,
    )
    for idx, panel in enumerate(prepared):
        row, col = divmod(idx, cols)
        x = padding + col * (cell_w + padding)
        y = padding + row * (cell_h + padding)
        canvas.paste(panel, (x, y))
    return canvas


def maybe_display(image):
    display(image)
    return image


def draw_box(image_array, box, *, colour=(220, 48, 48), width=3):
    image = np_to_pil(image_array).convert('RGB')
    draw = ImageDraw.Draw(image)
    x, y, w, h = box
    for offset in range(width):
        draw.rectangle((x - offset, y - offset, x + w + offset, y + h + offset), outline=colour)
    return image


def text_panel(lines, title, *, size=(560, 220)):
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    y = 38
    for line in lines:
        draw.text((12, y), line, fill=(20, 20, 20), font=FONT)
        y += 16
    return image


def bar_chart(values, labels, title, *, colours=None, size=(560, 280), value_fmt='{:.2f}'):
    if colours is None:
        colours = [(42, 157, 143)] * len(values)
    width, height = size
    left, right, top, bottom = 60, 20, 40, 50
    chart_w = width - left - right
    chart_h = height - top - bottom
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    max_value = max(max(values), 1e-6)
    bar_w = max(12, chart_w // max(len(values) * 2, 2))
    gap = bar_w
    x = left
    baseline = top + chart_h
    draw.line((left, top, left, baseline), fill=(60, 60, 60), width=1)
    draw.line((left, baseline, left + chart_w, baseline), fill=(60, 60, 60), width=1)
    for value, label, colour in zip(values, labels, colours, strict=True):
        bar_h = int((value / max_value) * (chart_h - 10)) if max_value > 0 else 0
        draw.rectangle((x, baseline - bar_h, x + bar_w, baseline), fill=colour)
        draw.text((x, baseline + 6), label[:12], fill=(20, 20, 20), font=FONT)
        draw.text((x, baseline - bar_h - 14), value_fmt.format(value), fill=(20, 20, 20), font=FONT)
        x += bar_w + gap
    return image


def grouped_bar_chart(groups, series_names, matrix, title, *, colours=None, size=(680, 300), value_fmt='{:.2f}'):
    if colours is None:
        colours = [(164, 68, 68), (43, 122, 120), (69, 123, 157)]
    width, height = size
    left, right, top, bottom = 60, 20, 40, 55
    chart_w = width - left - right
    chart_h = height - top - bottom
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    max_value = max(max(series) for series in matrix)
    baseline = top + chart_h
    draw.line((left, top, left, baseline), fill=(60, 60, 60), width=1)
    draw.line((left, baseline, left + chart_w, baseline), fill=(60, 60, 60), width=1)
    group_w = chart_w // max(len(groups), 1)
    series_count = len(series_names)
    bar_w = max(10, group_w // (series_count + 1))
    for group_idx, group in enumerate(groups):
        base_x = left + group_idx * group_w + 10
        for series_idx, _ in enumerate(series_names):
            value = matrix[series_idx][group_idx]
            bar_h = int((value / max(max_value, 1e-6)) * (chart_h - 12)) if max_value > 0 else 0
            x0 = base_x + series_idx * bar_w
            draw.rectangle((x0, baseline - bar_h, x0 + bar_w - 2, baseline), fill=colours[series_idx % len(colours)])
            draw.text((x0, baseline - bar_h - 12), value_fmt.format(value), fill=(20, 20, 20), font=FONT)
        draw.text((base_x, baseline + 6), group[:12], fill=(20, 20, 20), font=FONT)
    legend_x = width - 210
    for idx, name in enumerate(series_names):
        y = 24 + idx * 16
        draw.rectangle((legend_x, y, legend_x + 10, y + 10), fill=colours[idx % len(colours)])
        draw.text((legend_x + 16, y - 1), name, fill=(20, 20, 20), font=FONT)
    return image

### Synthetic industrial image generation code

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PartSample:
    sample_id: str
    label: str
    stamp: str
    variant: str
    image: np.ndarray


def render_panel(label, stamp, *, offset=(0, 0), scale=1.0, fixture=False):
    image = Image.new('RGB', (128, 128), (44, 52, 58))
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((20, 24, 108, 112), radius=6, fill=(94, 104, 106))
    cx, cy = 64 + offset[0], 64 + offset[1]
    half = max(15, round(19 * scale))
    left, top, right, bottom = cx - half, cy - half, cx + half, cy + half
    if label == 'normal':
        draw.ellipse((left, top, right, bottom), fill=(192, 208, 196), outline=(32, 38, 40), width=3)
    else:
        draw.rounded_rectangle((left, top, right, bottom), radius=2, fill=(188, 204, 196), outline=(32, 38, 40), width=3)
    if fixture:
        draw.line((24, 94, 104, 36), fill=(118, 126, 130), width=2)
    colours = {'red': (216, 70, 64), 'blue': (56, 116, 214), 'none': (44, 52, 58)}
    draw.rectangle((6, 6, 26, 26), fill=colours[stamp])
    return np.asarray(image, dtype=np.float32) / 255.0


def make_samples():
    specs = [
        ('normal', 'blue', 'aligned'),
        ('normal', 'blue', 'aligned'),
        ('normal', 'blue', 'aligned'),
        ('defect', 'red', 'aligned'),
        ('defect', 'red', 'aligned'),
        ('defect', 'red', 'aligned'),
        ('normal', 'red', 'swapped_stamp'),
        ('defect', 'blue', 'swapped_stamp'),
        ('normal', 'none', 'no_stamp'),
        ('defect', 'none', 'no_stamp'),
    ]
    return [
        PartSample(
            sample_id=f'part_{idx:02d}',
            label=label,
            stamp=stamp,
            variant=variant,
            image=render_panel(label, stamp, offset=((idx % 3) - 1, (idx % 2) * 2 - 1), fixture=idx % 2 == 0),
        )
        for idx, (label, stamp, variant) in enumerate(specs)
    ]

### Shortcut and intervention model code

In [ ]:
def stamp_shortcut_score(image):
    region = image[6:26, 6:26]
    return float(region[:, :, 0].mean() - region[:, :, 2].mean())


def shape_score(image):
    region = image[42:86, 42:86]
    occupied = np.linalg.norm(region - np.array([0.74, 0.80, 0.77]), axis=2) < 0.22
    row_counts = occupied.sum(axis=1)
    full_rows = np.count_nonzero(row_counts > 26)
    centre_column = np.count_nonzero(occupied[:, occupied.shape[1] // 2])
    return float(full_rows - 0.15 * centre_column - 18)


def predict(score):
    return 'defect' if score > 0 else 'normal'


def evaluate(samples, scorer):
    rows = []
    for sample in samples:
        predicted = predict(scorer(sample.image))
        rows.append({'sample': sample, 'predicted': predicted, 'correct': predicted == sample.label})
    return rows


def accuracy(rows, variant=None):
    selected = [row for row in rows if variant is None or row['sample'].variant == variant]
    return float(np.mean([row['correct'] for row in selected])) if selected else 0.0

### Build the dataset and run both models

In [ ]:
samples = make_samples()
shortcut_rows = evaluate(samples, stamp_shortcut_score)
intervention_rows = evaluate(samples, shape_score)
print('Shortcut overall accuracy:', accuracy(shortcut_rows))
print('Shortcut swapped-stamp accuracy:', accuracy(shortcut_rows, 'swapped_stamp'))
print('Intervention overall accuracy:', accuracy(intervention_rows))
print('Intervention swapped-stamp accuracy:', accuracy(intervention_rows, 'swapped_stamp'))

## Dataset and task definition
The synthetic part is either a `normal` disc or a `defect` block. During training-like aligned cases, a blue corner stamp appears with the normal class and a red stamp appears with the defect class.

That correlation is accidental. The true label is defined by the central part geometry.

In [ ]:
maybe_display(montage([titled(sample.image, f'{sample.label} | {sample.variant}') for sample in samples[:6]], cols=3))

## Model and explanation methods
The baseline model reads the corner stamp colour. The intervention reads the central shape. Both are explicit rules, so the explanation is the region each rule actually uses.

In [ ]:
maybe_display(montage([
    titled(draw_box(samples[0].image, (6, 6, 20, 20)), 'Shortcut evidence region'),
    titled(draw_box(samples[0].image, (42, 42, 44, 44)), 'Intervention evidence region'),
], cols=2))

## Baseline result
The shortcut model looks good on aligned examples, which is exactly why this failure mode is dangerous.

In [ ]:
maybe_display(montage([
    titled(row['sample'].image, f"{row['sample'].variant} | {row['sample'].label} -> {row['predicted']}")
    for row in shortcut_rows
], cols=5))

variants = ['aligned', 'swapped_stamp', 'no_stamp']
maybe_display(grouped_bar_chart(
    variants,
    ['Shortcut baseline', 'Shape intervention'],
    [[accuracy(shortcut_rows, variant) for variant in variants], [accuracy(intervention_rows, variant) for variant in variants]],
    'Accuracy by evaluation slice',
))

## Failure or pitfall
Once the corner stamp is swapped, the shortcut model follows the stamp rather than the part. The explanation is correspondingly bad: the red evidence box sits in the corner instead of on the object.

In [ ]:
swapped_normal = next(row['sample'] for row in shortcut_rows if row['sample'].sample_id == 'part_06')
swapped_defect = next(row['sample'] for row in shortcut_rows if row['sample'].sample_id == 'part_07')
maybe_display(montage([
    titled(draw_box(swapped_normal.image, (6, 6, 20, 20)), 'Shortcut evidence on swapped-stamp normal'),
    titled(draw_box(swapped_defect.image, (6, 6, 20, 20)), 'Shortcut evidence on swapped-stamp defect'),
], cols=2))

counter_panels = []
for sample in [swapped_normal, swapped_defect]:
    no_stamp = render_panel(sample.label, 'none')
    counter_panels.append(titled(sample.image, f'Original -> {predict(stamp_shortcut_score(sample.image))}'))
    counter_panels.append(titled(no_stamp, f'Removed stamp -> {predict(stamp_shortcut_score(no_stamp))}'))
maybe_display(montage(counter_panels, cols=2))

## Intervention
The intervention classifier reads the central part geometry instead. Its evidence box sits over the object, not the corner stamp.

In [ ]:
maybe_display(montage([
    titled(draw_box(samples[0].image, (42, 42, 44, 44)), 'Intervention evidence on the part region'),
    text_panel([
        f'part_06 shortcut -> {predict(stamp_shortcut_score(swapped_normal.image))}',
        f'part_06 intervention -> {predict(shape_score(swapped_normal.image))}',
        f'part_07 shortcut -> {predict(stamp_shortcut_score(swapped_defect.image))}',
        f'part_07 intervention -> {predict(shape_score(swapped_defect.image))}',
    ], 'Shortcut versus intervention on swapped-stamp cases'),
], cols=2))

## Re-test
The intervention keeps working when the stamp is swapped or removed, because the causal object evidence is still present.

In [ ]:
maybe_display(text_panel([
    f'Shortcut swapped-stamp accuracy: {accuracy(shortcut_rows, "swapped_stamp"):.2f}',
    f'Intervention swapped-stamp accuracy: {accuracy(intervention_rows, "swapped_stamp"):.2f}',
    f'Shortcut no-stamp accuracy: {accuracy(shortcut_rows, "no_stamp"):.2f}',
    f'Intervention no-stamp accuracy: {accuracy(intervention_rows, "no_stamp"):.2f}',
], 'Re-test summary'))

## What we learned
- Accidental production artefacts can look predictive enough to carry the whole model.
- The evidence view makes the failure obvious: the model is looking in the corner.
- A part-focused intervention restores robustness in the slices where the shortcut fails.

## Residual risks and next questions
- Real industrial shortcuts can be subtler than a coloured stamp.
- The intervention here is a hand-crafted rule rather than a trained model.
- A stronger version would compare learned saliency maps and exemplar retrieval on real data.